# FireSystem: Burn / No-Burn Training Dataset and Monthly Series
#### This notebook uses the helpers in `src/utils/appeears.py` (AppEEARS API client) and `src/utils/fire_datasets.py` (dataframe pipeline) and `src/utils/chile_regions.py` (Chilean region boundaries) to:

1. Request burned area, vegetation indices and land cover for one or more Chilean regions (each region's outline, simplified to at most 50 points) as NetCDF4 on a geographic (lat/lon) grid.
2. Build one dataframe per dataset, each with date, `lat`/`lon` and pixel bounding-box columns, and a description (`df.attrs["description"]`) with update frequency and spatial granularity.
3. Add no-burn samples: for every burned pixel and year, one random day and location in each month without a burn.
4. Attach vegetation (NDVI/EVI) and land cover to every burn and no-burn row: the pixel's **last valid measurement before the date**, taken as constant. Its observation period must end at least `burn_date_uncertainty` + 1 day before the date, so nothing observed on or after the fire is used (no data leakage); `veg_age_days` says how old that measurement is.
5. Combine burned area (label) and vegetation (features) into a training dataset.
6. Combine the regions: each region contributes burn samples in proportion to its number of fires (and, within the region, each year and month in proportion to its own fires).
7. Build the regional monthly time series (one row per region and month: fires, burned pixels, NDVI/EVI) for time-series models.

Every dataframe is clipped to the exact region outline: only pixels whose centre is inside the region, and random no-burn locations inside the region, are kept, so no data from Argentina, Bolivia, Peru or the ocean is used.

| Dataset | Product | Update frequency | Spatial granularity | Available from |
|---|---|---|---|---|
| MODIS burned area | `MCD64A1.061` | Monthly (rows dated by day of burn) | 500 m | 2000-11-01 |
| VIIRS burned area | `VNP64A1.002` | Monthly (rows dated by day of burn) | 500 m | 2012-03-01 |
| MODIS vegetation (NDVI/EVI) | `MOD13A1.061` | 16-day composite | 500 m | 2000-02-18 |
| VIIRS vegetation (NDVI/EVI) | `VNP13A1.002` | 16-day composite | 500 m | 2012-01-17 |
| MODIS land cover | `MCD12Q1.061` | Yearly | 500 m | 2001-01-01 |

VIIRS datasets are requested only when the time range reaches their start date.

Region boundaries: [geoBoundaries Chile ADM1](https://www.geoboundaries.org/api/current/gbOpen/CHL/ADM1/) (simplified, 2020; source Biblioteca del Congreso Nacional de Chile / OCHA ROLAC; CC BY 3.0 IGO), stored in `data/boundaries/geoBoundaries-CHL-ADM1_simplified.geojson`.

---
## 1. Set Up
#### Add the repository root to the import path and import the FireSystem helpers.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import earthaccess

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import appeears
from src.utils import chile_regions as cr
from src.utils import fire_datasets as fd

---
## 2. Set Request Parameters
#### Set `REGION` in the first cell below to the region to run (one of the names in `CHILE_REGIONS`, e.g. `"Maule"`, `"Ñuble"`, `"Biobío"`, `"O'Higgins"`). Its data is requested separately and stored in its own folder under `data_dir`; the output files are named after it. The time range runs from 2012-03-01 (start of VIIRS burned area) to today.
#### Burn sampling is strictly proportional: a region, year and month with `f` of the `F` fires in the time range gets `MAX_BURN_SAMPLES × f / F` burn samples (rounded so the total is exactly `MAX_BURN_SAMPLES`, never more than its burned pixels). `MAX_BURN_SAMPLES` is derived from `MAX_TOTAL_GB` (downloads + output files); no-burn samples are then generated only for the drawn pixels.

In [ ]:
# Region to run: change this for the next region (e.g. "Ñuble", "Biobío", "O'Higgins",
# or "Maule2": a 1/10-area part of Maule for a quick, small run)
REGION = "Maule4"

In [ ]:
CHILE_REGIONS = {
    "Arica y Parinacota":   (-70.38, -19.23, -68.91, -17.49),
    "Tarapacá":             (-70.29, -21.64, -68.40, -18.93),
    "Antofagasta":          (-70.74, -26.07, -66.99, -20.93),
    "Atacama":              (-71.53, -29.54, -68.26, -25.28),
    "Coquimbo":             (-71.72, -32.29, -69.80, -29.03),
    "Valparaíso":           (-71.85, -33.96, -69.98, -32.02),  # mainland only
    "Metropolitana":        (-71.72, -34.30, -69.76, -32.92),
    "O'Higgins":            (-72.06, -35.01, -70.00, -33.85),
    "Maule":                (-72.79, -36.55, -70.31, -34.68),
    "Ñuble":                (-72.89, -37.20, -71.00, -36.00),
    "Biobío":               (-73.97, -38.50, -70.98, -36.44),
    "La Araucanía":         (-73.52, -39.64, -70.82, -37.58),
    "Los Ríos":             (-73.73, -40.69, -71.59, -39.28),
    "Los Lagos":            (-74.85, -44.07, -71.58, -40.23),
    "Aysén":                (-75.66, -49.22, -71.09, -43.63),
    "Magallanes":           (-75.73, -55.99, -66.41, -48.59),  # excludes the Antarctic claim
}

# Reduced regions: name -> (parent region, share of its area kept around its centre)
REDUCED_REGIONS = {
    "Maule2": ("Maule", 0.1),
    "Maule3": ("Maule", 0.2),
    "Maule4": ("Maule", 0.05),
}

if REGION not in CHILE_REGIONS and REGION not in REDUCED_REGIONS:
    raise ValueError(f"Unknown REGION {REGION!r}; choose one of: {', '.join([*CHILE_REGIONS, *REDUCED_REGIONS])}")
REGIONS = [REGION]

start = datetime(2012, 3, 1)     # start of the date range (VIIRS burned area starts 2012-03-01)
stop = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)  # end of the date range: today
data_dir = REPO_ROOT / "data" / "AppEEARS"

# Per region: bbox, exact outline (for clipping), <= 50-point covering outline (for the request), folder
chile = cr.load_regions(REPO_ROOT / "data" / "boundaries" / cr.BOUNDARIES_FILE)
regions = {}
for name in REGIONS:
    if name in REDUCED_REGIONS:
        parent, fraction = REDUCED_REGIONS[name]
        geom = cr.shrink_geometry(cr.region_geometry(chile, parent), fraction)
        bbox = cr.geometry_bbox(geom)
    else:
        geom = cr.region_geometry(chile, name)
        bbox = CHILE_REGIONS[name]
    regions[name] = {
        "bbox": bbox,
        "geometry": geom,
        "request": cr.request_geometry(geom, max_points=50),
        "directory": data_dir / cr.region_slug(name),
    }
    print(f"{name:20} {cr.geometry_area_km2(geom):,.0f} km²  bbox {regions[name]['bbox']}  outline {cr.polygon_points(regions[name]['request'])} points  -> {regions[name]['directory']}")

SEED = 42                        # random seed for no-burn dates and locations
EXTENT = "pixel"                 # no-burn locations: "pixel" (inside the burned pixel) or "study_area" (anywhere in bbox)
SUBMIT_NEW_TASK = True           # False: reuse NetCDF files already in data_dir

MAX_TOTAL_GB = 15                # budget for downloads + output files; sets MAX_BURN_SAMPLES below
SAMPLE_BY = ("year", "month")    # periods used for the fire ratio, e.g. ("year",) for yearly only
FIRE_COUNT = "events"            # "events": adjacent pixels burned <= MAX_GAP_DAYS apart = 1 fire; "pixels": burned pixels
MAX_GAP_DAYS = 5
REDISTRIBUTE = False             # False: strict proportions (MAX x fires / total fires); True: pass unused shares on
OUTPUT_FORMAT = "csv"            # "csv" (+ .description.json with the description) or "json" (one file)
KEEP_DATAFRAMES = len(REGIONS) == 1  # keep every region's dataframes for inspection (uses more memory)

KEEP = ("burned_area", "vegetation", "land_cover")
datasets = {k: v for k, v in fd.active_datasets(stop).items() if k.endswith(KEEP)}
for name, d in datasets.items():
    desc = d["description"]
    print(f"{name:20} {d['product']:14} {desc['update_frequency']:18} {desc['spatial_granularity']}")

# Size budget: estimated raw download of every region's request, the rest for the output files
download_bytes = sum(
    fd.estimate_download_bytes(cr.geometry_area_km2(spec["request"]), start, stop, datasets) for spec in regions.values()
)
n_sensors = sum(name.endswith("burned_area") for name in datasets)
output_budget = MAX_TOTAL_GB * 1e9 - download_bytes
MAX_BURN_SAMPLES = fd.max_burn_samples_for_bytes(output_budget, sensors=n_sensors)
print(f"\nEstimated raw download: {download_bytes / 1e9:.1f} GB | output budget: {output_budget / 1e9:.1f} GB "
      f"| MAX_BURN_SAMPLES: {MAX_BURN_SAMPLES:,} per sensor")

---
## 3. Submit, Wait for and Download the Requests
#### One AppEEARS request per region: all are submitted first, then each is awaited and downloaded into the region's folder. Earthdata Login credentials are read from `~/.netrc` (created by `earthaccess.login` if missing). Large areas or long time ranges can take a while to process.

In [ ]:
if SUBMIT_NEW_TASK:
    earthaccess.login(strategy="interactive", persist=True)
    fire_headers = appeears.login(*appeears.credentials_from_netrc())

    fire_task_ids = {}
    for name, spec in regions.items():
        fire_task = appeears.build_area_task(
            name=f"firesystem_{cr.region_slug(name)}_{start:%Y%m%d}_{stop:%Y%m%d}",
            bbox=spec["bbox"],
            start=start,
            stop=stop,
            layers=fd.request_layers(datasets),
            geo=cr.geometry_to_geojson(spec["request"], name=name),
        )
        fire_task_ids[name] = appeears.submit_task(fire_headers, fire_task)
        print(f"{name}: task {fire_task_ids[name]}")

    for name, task_id in fire_task_ids.items():
        appeears.wait_for_task(fire_headers, task_id)
        fire_files = appeears.download_bundle(fire_headers, task_id, regions[name]["directory"])
        print(f"{name}: {len(fire_files)} files in {regions[name]['directory']}")
    appeears.logout(fire_headers)

---
## 4. Build the Datasets
#### `fd.build_multi_region_dataset` counts the fires of every region, allocates `MAX_BURN_SAMPLES` across regions, years and months in proportion to them, builds each region's samples with its allocation and combines the training rows. `region_summary` shows, per region and sensor, the fires, the region's share (`ratio`), the burn samples allocated and the rows that reached the training dataset. `fd.build_regional_monthly_series` builds the monthly time series (one row per region and month).

In [ ]:
result = fd.build_multi_region_dataset(
    regions, start, stop,
    max_burn_samples=MAX_BURN_SAMPLES, seed=SEED, extent=EXTENT, datasets=datasets,
    fire_count=FIRE_COUNT, max_gap_days=MAX_GAP_DAYS, sample_by=SAMPLE_BY, redistribute=REDISTRIBUTE,
    keep_dataframes=KEEP_DATAFRAMES,
)

series = fd.build_regional_monthly_series(
    regions, start, stop, datasets=datasets, fire_count=FIRE_COUNT, max_gap_days=MAX_GAP_DAYS,
)
print(f"Monthly series: {len(series):,} rows ({series['region'].nunique()} regions x {series['date'].nunique()} months)")

region_summary = result["region_summary"]
print(region_summary.attrs["description"]["variables"])
region_summary

#### Fires per region and period, ratio (share of the region's fires), burned pixels available, burn samples drawn and no-burn samples generated. Periods with only no-burn samples have 0 fires.

In [ ]:
fire_summary = result["fire_summary"]
print("Fires:", fire_summary["fires"].sum(), "| burn samples:", fire_summary["burn_samples"].sum(),
      "| no-burn samples:", fire_summary["no_burn_samples"].sum())
fire_summary

In [ ]:
# Inspect one region's dataframes (kept when KEEP_DATAFRAMES = True)
dfs = result["dataframes"].get(REGIONS[0])
if dfs is None:
    print("Set KEEP_DATAFRAMES = True in section 2 to inspect each region's dataframes.")
else:
    for name, df in dfs.items():
        desc = df.attrs["description"]
        print(f"{name:36} rows={len(df):>9,}  {desc.get('update_frequency', '')}  |  {desc.get('spatial_granularity', '')}")
    burned_area = dfs["MODIS_burned_area"]
    print(burned_area.attrs["description"])
    display(burned_area.head())

In [ ]:
if dfs is not None:
    vegetation_at_burn_dates = dfs["MODIS_vegetation_at_burn_dates"]
    print(vegetation_at_burn_dates.attrs["description"])
    display(vegetation_at_burn_dates.head())

---
## 5. Training Dataset, Monthly Series and Saving
#### Training dataset: features `year`, `quarter`, `month`, `day`, `lat`, `lon`, `ndvi`, `evi` (last valid measurement before the date), `veg_age_days`; label `burned`; `region` and `sensor` identify the Chilean region and MODIS or VIIRS rows. Rows without NDVI and EVI are dropped.
#### Monthly series: one row per region and month with `<SENSOR>_fires`, `<SENSOR>_burned_pixels` (targets) and `<SENSOR>_ndvi_mean` / `_evi_mean` from the last composite that ended on or before the 1st of the month (plus spatial standard deviation, valid share and `_veg_age_days`), so a month's vegetation never includes data from that month.
#### Both are saved to `data_dir` as CSV (plus a `.description.json`) or JSON, depending on `OUTPUT_FORMAT`; the total size of downloads and outputs is checked against `MAX_TOTAL_GB`.

In [ ]:
training = result["training"]
print(training.attrs["description"])
print(training.groupby(["region", "burned"]).size().unstack(fill_value=0))
display(training.head())

print(series.attrs["description"]["variables"])
series.head(12)

In [ ]:
# Save only once the datasets are filled
regions_label = cr.region_slug(REGIONS[0]) if len(REGIONS) == 1 else f"{len(REGIONS)}_regions"
period_label = f"{start:%Y%m%d}_{stop:%Y%m%d}"
saved = fd.save_dataframe(series, data_dir / f"monthly_series_{regions_label}_{period_label}", OUTPUT_FORMAT)
if training.empty:
    print("Training dataset is empty (no burns in the region and time range): not saved.")
else:
    saved += fd.save_dataframe(training, data_dir / f"training_{regions_label}_{period_label}", OUTPUT_FORMAT)
print("Saved:", *saved, sep="\n  ")

# Check the total size of downloads and outputs against the budget
used = sum(f.stat().st_size for spec in regions.values() for f in Path(spec["directory"]).glob("*")) + sum(f.stat().st_size for f in saved)
print(f"Downloads + outputs on disk: {used / 1e9:.2f} GB of {MAX_TOTAL_GB} GB")